In [1]:
import pandas as pd

df = pd.read_csv("relevance_validation.csv")

df.head()

,uuid,title,maintext,jeremy_3way,julia_3way,jeremy_binary,julia_binary,consensus_3way
0,72d695f7-7ef5-4b3f-a1ed-fa9a4148cd21,"Putin running for reelection, almost sure to w...",Vladimir Putin on Friday moved to prolong his ...,0,0,0,0,0
1,1b25ea37-f2b8-452e-aec0-1e321235b05c,Proud Boys leader Enrique Tarrio found guilty ...,"Enrique Tarrio, leader of the Proud Boys, in O...",1,1,1,1,1
2,fdd46e8b-5263-49f8-8040-39abf87a89e3,Is The Rally Running Out Of Steam?,Gold prices are up 9% since February 14. (Phot...,0,0,0,0,0
3,a6ee042c-f271-4e17-8c78-5c4c705387fa,Google fires 28 employees after protest agains...,Protests across the U.S. call for Gaza cease-f...,2,2,1,1,2
4,a5160e01-1eae-4f6c-b52c-9fcd9bfcabff,Study finds 'no evidence' military veterans le...,Veterans are generally less likely than Americ...,1,1,1,1,1


In [2]:
import os
import json
import re
import time
import random
from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from openai import OpenAI

LABELS = ["IRRELEVANT", "RELEVANT_DIRECT", "RELEVANT_INDIRECT"]

GOLD_INT_TO_LABEL = {
    0: "IRRELEVANT",
    1: "RELEVANT_INDIRECT",
    2: "RELEVANT_DIRECT",
}

def normalize_gold(consensus_3way):
    v = int(consensus_3way)
    return GOLD_INT_TO_LABEL[v]


def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None

    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None


def normalize_pred(label):
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s:
        return "IRRELEVANT"
    if "indirect" in s:
        return "RELEVANT_INDIRECT"
    if "direct" in s:
        return "RELEVANT_DIRECT"
    return None


In [8]:
import hashlib
import json

@dataclass
class GPTCallConfig:
    model: str = "gpt-4o-mini"
    temperature: float = 0.0
    max_output_tokens: int = 80
    max_retries: int = 5


def backoff_sleep(attempt):
    base = min(2 ** attempt, 30)
    time.sleep(base + random.uniform(0, 0.25 * base))



CACHE_PATH = "gpt_cache.json"

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r") as f:
        GPT_CACHE = json.load(f)
else:
    GPT_CACHE = {}

def call_gpt_label(client, prompt_spec, title, text, cfg):
    # --- CACHE KEY (make it depend on anything that would change the output) ---
    user_msg = prompt_spec["user_template"].format(title=title, text=text)

    key_payload = {
        "prompt_name": prompt_spec.get("name", ""),
        "system": prompt_spec.get("system", ""),
        "user_msg": user_msg,
        "model": cfg.model,
        "temperature": cfg.temperature,
        "max_output_tokens": cfg.max_output_tokens,
    }
    key_str = json.dumps(key_payload, sort_keys=True, ensure_ascii=False)
    cache_key = hashlib.sha256(key_str.encode("utf-8")).hexdigest()

    # --- RETURN CACHED RESULT IF PRESENT ---
    if cache_key in GPT_CACHE:
        cached = GPT_CACHE[cache_key]
        return cached["pred_label"], cached["raw_output"]

    # --- OTHERWISE CALL THE API ---
    for attempt in range(cfg.max_retries):
        try:
            resp = client.responses.create(
                model=cfg.model,
                input=[
                    {"role": "system", "content": prompt_spec["system"]},
                    {"role": "user", "content": user_msg},
                ],
                temperature=cfg.temperature,
                max_output_tokens=cfg.max_output_tokens,
            )

            out_text = (resp.output_text or "").strip()

            obj = extract_first_json_object(out_text)
            if obj and "label" in obj:
                pred = normalize_pred(obj["label"])
            else:
                pred = None
                for lab in LABELS:
                    if lab in out_text.upper():
                        pred = lab
                        break

            # --- SAVE TO CACHE (even if pred is None; helps debugging) ---
            GPT_CACHE[cache_key] = {
                "pred_label": pred,
                "raw_output": out_text,
            }
            with open(CACHE_PATH, "w") as f:
                json.dump(GPT_CACHE, f, ensure_ascii=False)

            return pred, out_text

        except Exception as e:
            if attempt == cfg.max_retries - 1:
                # Cache the error too, so reruns don't spam calls if something is persistently failing
                err_text = f"ERROR: {repr(e)}"
                GPT_CACHE[cache_key] = {
                    "pred_label": None,
                    "raw_output": err_text,
                }
                with open(CACHE_PATH, "w") as f:
                    json.dump(GPT_CACHE, f, ensure_ascii=False)
                return None, err_text

            backoff_sleep(attempt)


In [9]:
#df = pd.read_csv("relevance_40.csv").reset_index(drop=True)
df["gold_label"] = df["consensus_3way"].apply(normalize_gold) # make numerical

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(df, df["gold_label"]))


In [10]:
def run_oof_for_prompt(df, folds, prompt_spec, client, cfg):
    n = len(df)
    preds = [None] * n
    raws = [""] * n
    fold_ids = [-1] * n

    for fold_id, (_, test_idx) in enumerate(folds):
        for idx in tqdm(test_idx, desc=f"{prompt_spec['name']} fold {fold_id+1}", leave=False):
            p, raw = call_gpt_label(
                client,
                prompt_spec,
                df.loc[idx, "title"],
                df.loc[idx, "maintext"],
                cfg
            )
            preds[idx] = p
            raws[idx] = raw
            fold_ids[idx] = fold_id

    out = df.copy()
    out["prompt_name"] = prompt_spec["name"]
    out["fold_id"] = fold_ids
    out["pred_label"] = preds
    out["raw_output"] = raws
    out["is_valid_pred"] = out["pred_label"].isin(LABELS)
    return out

def compute_metrics(oof_df):
    valid = oof_df[oof_df["is_valid_pred"]]
    y_true = valid["gold_label"]
    y_pred = valid["pred_label"]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=LABELS),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABELS)
    }


In [13]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import importlib
import relevance_prompts
importlib.reload(relevance_prompts)

PROMPT_VARIANTS = relevance_prompts.PROMPT_VARIANTS

load_dotenv() 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

cfg = GPTCallConfig(
    model="gpt-4o-mini",
    temperature=0.0,
    max_output_tokens=80
)

all_results = []
os.makedirs("results", exist_ok=True)

for prompt in PROMPT_VARIANTS:
    print(f"\n=== {prompt['name']} ===")
    oof = run_oof_for_prompt(df, folds, prompt, client, cfg)
    metrics = compute_metrics(oof)

    print("Accuracy:", metrics["accuracy"])
    print("Macro F1:", metrics["macro_f1"])
    print(metrics["confusion_matrix"])

    oof.to_csv(f"results/{prompt['name']}_oof.csv", index=False)
    all_results.append({
        "prompt_name": prompt["name"],
        **metrics
    })

summary = pd.DataFrame(all_results).sort_values("macro_f1", ascending=False)
summary.to_csv("results/relevance_summary_metrics.csv", index=False)
summary



=== relevance_v1_zero_shot_minimal ===


Accuracy: 0.65
Macro F1: 0.6198257080610022
[[ 4  4  5]
 [ 0  9  2]
 [ 0  3 13]]

=== relevance_v2_zero_shot_two_stage_internal ===


Accuracy: 0.675
Macro F1: 0.6594986290638464
[[ 7  4  2]
 [ 0  6  5]
 [ 0  2 14]]

=== relevance_v3_few_shot_trimmed_with_reasons ===


Accuracy: 0.675
Macro F1: 0.6572230124861703
[[ 6  4  3]
 [ 0  7  4]
 [ 0  2 14]]

=== relevance_v4_few_shot_stricter_irrelevant_guardrail ===


Accuracy: 0.675
Macro F1: 0.6765142464067195
[[ 8  4  1]
 [ 0  8  3]
 [ 1  4 11]]



=== relevance_v5_zero_shot_codebook_style_more_explicit ===


Accuracy: 0.675
Macro F1: 0.6572230124861703
[[ 6  4  3]
 [ 0  7  4]
 [ 0  2 14]]

=== relevance_v6_two_field_output_with_rationale_tags ===


Accuracy: 0.55
Macro F1: 0.5611010284331518
[[6 5 2]
 [0 8 3]
 [0 8 8]]


,prompt_name,accuracy,macro_f1,confusion_matrix
3,relevance_v4_few_shot_stricter_irrelevant_guar...,0.675,0.676514,"[[8, 4, 1], [0, 8, 3], [1, 4, 11]]"
1,relevance_v2_zero_shot_two_stage_internal,0.675,0.659499,"[[7, 4, 2], [0, 6, 5], [0, 2, 14]]"
4,relevance_v5_zero_shot_codebook_style_more_exp...,0.675,0.657223,"[[6, 4, 3], [0, 7, 4], [0, 2, 14]]"
2,relevance_v3_few_shot_trimmed_with_reasons,0.675,0.657223,"[[6, 4, 3], [0, 7, 4], [0, 2, 14]]"
0,relevance_v1_zero_shot_minimal,0.650,0.619826,"[[4, 4, 5], [0, 9, 2], [0, 3, 13]]"
5,relevance_v6_two_field_output_with_rationale_tags,0.550,0.561101,"[[6, 5, 2], [0, 8, 3], [0, 8, 8]]"
